In [25]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from gspread_pandas import Spread, conf

In [26]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427"
sales_sheet_url = "https://docs.google.com/spreadsheets/d/1h7txdIo1ha4vp2uE8W2E4uUXgJYSG_wgxyKGpW7rYAQ/edit?gid=1442776849#gid=1442776849"


# Load the config from your service account JSON
c = conf.get_config(file_name=config_path)

# Connect using that config object
sales_spread = Spread(sales_sheet_url, config=c)
emarath_spread = Spread(emarath_global_sheet_url, config=c)

In [27]:
# Data from S10 sale sheet
target_statuses = ["Super Hot","Hot","Cold","Won"]
ignore_sheets = ["Chart", "BDE Sales Activity", "Select BDE/STATUS", "Won", "Super Hot", "Hot", "Cold", "Headers,Dropdown data etc"]

all_data = []

for sheet in sales_spread.sheets:
    sheet_name = sheet.title
    if getattr(sheet, 'sheet_type', '') == 'CHART' or sheet_name in ignore_sheets:
        continue

    try:
        # Start from row 10 where the headers are
        df = sales_spread.sheet_to_df(sheet=sheet_name, index=None, start_row=10)
        df.columns = df.columns.str.strip()
        
        if not df.empty and all(col in df.columns for col in ['Timestamp', 'Name of BDE', 'STATUS']):
        
            df['Date'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce').dt.date
            
            filtered_df = df[df['STATUS'].isin(target_statuses)].copy()
            
            if not filtered_df.empty:
                all_data.append(filtered_df[['Date', 'Name of BDE', 'STATUS']])
                
    except Exception as e:
        print(f"Skipping {sheet_name}: {e}")

if all_data:
    master_df = pd.concat(all_data, ignore_index=True)
    
    summary = master_df.groupby(['Date', 'Name of BDE', 'STATUS']).size().reset_index(name='Count')
    
    pivot_summary = summary.pivot_table(
        index=['Date', 'Name of BDE'], 
        columns='STATUS', 
        values='Count', 
        aggfunc='sum'
    ).fillna(0).astype(int)

    print("\n--- DAILY BDE PERFORMANCE ---")
    print(pivot_summary)
else:
    print("No data found matching the criteria.")

Skipping Select Date: Can only use .str accessor with string values, not integer
Skipping Sales Chart: {'code': 400, 'message': "Invalid range: 'Sales Chart'", 'status': 'INVALID_ARGUMENT'}
Skipping Sheet14: Can only use .str accessor with string values, not integer
Skipping Sheet15: Can only use .str accessor with string values, not integer
Skipping Unit Price: Can only use .str accessor with string values, not integer

--- DAILY BDE PERFORMANCE ---
STATUS                   Cold  Hot  Super Hot  Won
Date       Name of BDE                            
2026-01-30 Amina            0    0          4    4
           Arun            20    2          6    4
           Burhana          2    4          2    0
           Farsana         20    4          4    2
           Habiya (122)     2    0          4    0
           Rahib            0    2          4    2
2026-01-31 Amina            6    6          0    6
           Arun            28    4          4    6
           Burhana          0    2 

In [ ]:
# 5. Loop through and pull categorized data
all_data_list = []
all_sheets = emarath_spread.sheets

# Define exactly which statuses you want to track
target_statuses = ["SUPER HOT","HOT","WARM","COLD","WON","BOOKING", "WHATSAPP ENGAGE","UNATTENDED"]

for sheet in all_sheets:
    sheet_name = sheet.title
    
    # Skip dashboard and administrative tabs
    if sheet_name in ["MAIN DASHBOARD", "CRM", "Dropdowns", "Sheet1", "DASHBOARD", "20"]:
        continue
    
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        if not df.empty and all(col in df.columns for col in ['DATE', 'STATUS']):
            filtered_df = df[df['STATUS'].isin(target_statuses)].copy()
            
            if not filtered_df.empty:
                filtered_df['Name of BDE'] = sheet_name
                all_data_list.append(filtered_df[['DATE', 'Name of BDE', 'STATUS']])
            
    except Exception as e:
        print(f"Skipping {sheet_name}: {e}")

# Combine and Process
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)

    master_df['Date'] = pd.to_datetime(master_df['DATE'], dayfirst=True, errors='coerce').dt.date

    report_df = master_df.pivot_table(
        index=['Date', 'Name of BDE'], 
        columns='STATUS', 
        values='Count', 
        aggfunc='sum',
        fill_value=0
    )
    # Reorder status columns
    report_df = report_df.reindex(columns=target_statuses, fill_value=0)

    final_report = pd.concat([report_df], axis=1).fillna(0).astype(int)

    print("\n--- DAILY BDE PERFORMANCE (STATUS & LANGUAGE) ---")
    # print(final_report)
else:
    print("No data found matching the criteria.")


--- DAILY BDE PERFORMANCE (STATUS & LANGUAGE) ---


In [29]:
final_report

SUPER HOT  HOT  WARM  COLD  WON  BOOKING  \
Date       Name of BDE                                             
2026-02-04 ADHIL                0    0     0     0    0        0   
           AMINA                0    0     0     0    0        0   
           ARUN                 0    0     0     0    0        0   
           BURHANA              0    0     0     0    0        0   
           CHAITHANYA           0    0     0     0    0        0   
...                           ...  ...   ...   ...  ...      ...   
2026-02-16 RAHIB                0    0     0     0    0        0   
           RINSIYA              0    0     0     0    0        0   
           SAFAN                0    0     0     0    0        0   
           SHABNA               0    0     0     0    0        0   
           SHAMNA               0    0     0     0    0        0   

                        WHATSAPP ENGAGE  UNATTENDED  
Date       Name of BDE                               
2026-02-04 ADHIL                      0           0  
           AMINA                      0           0  
           ARUN                       0           0  
           BURHANA                    0           0  
           CHAITHANYA                 0           0  
...                                 ...         ...  
2026-02-16 RAHIB                      0           0  
           RINSIYA                    0           0  
           SAFAN                      0           0  
           SHABNA                     0           0  
           SHAMNA                     0           0  

[132 rows x 8 columns]

In [30]:
pivot_summary.columns = pivot_summary.columns.str.upper()
final_combined_report = pd.concat([pivot_summary, report_df], axis=0)
final_combined_report = final_combined_report.fillna(0).astype(int)
final_columns = ["WON","SUPER HOT","HOT","WARM","COLD","BOOKING", "WHATS APP ENGAGE","UNATTENDED"]
final_combined_report = final_combined_report.reindex(columns=final_columns, fill_value=0)

In [31]:
final_combined_report

WON  SUPER HOT  HOT  WARM  COLD  BOOKING  \
Date       Name of BDE                                              
2026-01-30 Amina           4          4    0     0     0        0   
           Arun            4          6    2     0    20        0   
           Burhana         0          2    4     0     2        0   
           Farsana         2          4    4     0    20        0   
           Habiya (122)    0          4    0     0     2        0   
...                      ...        ...  ...   ...   ...      ...   
2026-02-16 RAHIB           0          0    0     0     0        0   
           RINSIYA         0          0    0     0     0        0   
           SAFAN           0          0    0     0     0        0   
           SHABNA          0          0    0     0     0        0   
           SHAMNA          0          0    0     0     0        0   

                         WHATS APP ENGAGE  UNATTENDED  
Date       Name of BDE                                 
2026-01-30 Amina                        0           0  
           Arun                         0           0  
           Burhana                      0           0  
           Farsana                      0           0  
           Habiya (122)                 0           0  
...                                   ...         ...  
2026-02-16 RAHIB                        0           0  
           RINSIYA                      0           0  
           SAFAN                        0           0  
           SHABNA                       0           0  
           SHAMNA                       0           0  

[173 rows x 8 columns]

In [32]:
# Save the final combined report to an Excel file
file_name = "./Output_Data/tag_wise_Report.xlsx"
final_combined_report.to_excel(file_name, index=True)
print(f"Success! The report has been saved as: {file_name}")

Success! The report has been saved as: ./Output_Data/tag_wise_Report.xlsx


In [33]:

Dest_sheet_url = "https://docs.google.com/spreadsheets/d/1PqMNtFU0bas_BGYFhIYWojO2petW-TOGxeRB2OWnF08/edit?pli=1&gid=788610855#gid=788610855"
Dest_tab_name = "Tag_Wise_Report"

try:
    dest_spread = Spread(Dest_sheet_url, config=c)
    dest_spread.open_sheet(Dest_tab_name, create=False)

    export_df = final_combined_report.reset_index()

 
    export_df['Date'] = export_df['Date'].astype(str)
    export_df.loc[export_df['Date'].duplicated(), 'Date'] = ""

    existing_df = dest_spread.sheet_to_df()
    is_empty = existing_df.empty

    current_data_rows = len(existing_df)

    start_row = 1 if is_empty else current_data_rows + 2

    # 4. Push to sheet
    dest_spread.df_to_sheet(
        export_df,          
        index=False, 
        replace=False, 
        headers=is_empty,     
        start=f'A{start_row}' 
    )

    print(f"Cleaned report appended to '{Dest_tab_name}' starting at row {start_row}.")

except Exception as e:
    print(f"Detailed Error: {e}")

Cleaned report appended to 'Tag_Wise_Report' starting at row 173.
